<b> Title: </b> Understanding Performance Dynamics in Modern Formula 1

<b> Topic: </b> Analysing the relationship between qualifying performance and race outcomes, intra-team competitive dynamics, and strategic tyre management using data from 2018 to 2024 from the datasource FastF1. 

<b> Research Question 1: </b>  Qualifying-Race Performance Correlation

<u> “What is the relationship between qualifying session performance metrics and final race standings across different circuits, seasons, and regulatory eras?” </u>

<u> <i> <center> 1a) Does the qualifying-race relationship change depending on the context of circuit type and overtaking difficulty? </center> </i> </u>

<u> <i> <center> 1b) Do the regulations affecting overtaking introduced in 2022 affect the qualifying-race correlation? </center> </i> </u>

This notebook studies the above questions using various statistical analyses, further detailed in the table of contents below. The aim of this notebook is to aid in producing a report that is well supported by data. We do this through consturcting two datasets, one for an elastic net multinomial logistic regression and another for a time series analysis which is implemented in R. 

Some of the <b> key outputs </b> of this report are descriptive EDA graphs, cross-validated model performance metrics, subgroup comparisons, and within-rice dependence summaries. For transparency, I have tried to thoroguhly explain and justify the use of each statistical tool within the notebook, but the overall conclusion and further discussion in implemented in the report that this notebook complements. 

# insert table of contents here, need to fix formatting later. also table of outputs? to make it more professional
Including notes for my own clarity

1. Enivronment Setup // 1a) installing fastf1, turning off warnings, f1 cache enabling 1b) importing packages 1c) defining for reproducibility,  years we use in our data, year split 2022, how we define outcome (three classes), Qualifying and Race (Q and F).


2. Data Construction for question A // 
    2a) event-driver dataset for multinomial regression, whcih variables im stroting (variable table) and how i'm filtering, best qualifying time Q3 if exists else Q2 else Q1. merging data. derived labels (pre-2022, post 2022, circuit type(technical,non-tecnical etc), handling missing data, checking duplicates, total rows are comprehensible eda.  
    
3) EDA for q1A
    3a) class balance, qualifyig distrobution vs race finishing distribution, relationship plots, SPEARMAN Rank correlation, by circuit type and era etc. or is that q1b?
    
4) Analysis for q1A - Elastic net multinomial logistic regression
    4a) train test split, define seed and splits etc
    4b) preprocessing pipeline +SGD classifier
    4c) Cros validation for hyperparameter tuning and saving the best model
    4d) evaluation (accuracy, macro/weighted-f1, confusion matrix, classification report table. non-zero coeffs?
    4e) Subgroup eval? model performance by era (pre-2022). eval by circuitt type. add interaction terms?
    
5) results discussion. Furthermore, anoher way to evaluate can be looking at the lap by lap performance, using position as well whihc is dependent on qualifying performance

6) analysis within race time series (USING R)
....

    

extra, where does it belong?
defining seeds? train test splits? cv stuff.

Furthermore. I can use multinomial to look at both questions and then use time series to look at both questions a well. so divide the report into multinomial logistic and time series analysis. 

# Introduction?
to the two question and overall?

# 1. Enivronment Setup
This is evrything that is necessary to run the code. 

#### FastF1
First is downloading the data itself which is very simple through !pip install. Then I ensure local caching so that the downloaded sessions are stored and reused to make the code efficient. I'm also turning off warnings for a cleaner output.

In [5]:
# !pip install fastf1

import fastf1

import os 
os.makedirs("fastf1_cache", exist_ok=True)
fastf1.Cache.enable_cache("fastf1_cache") # enabling cache locally

# ignoring warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

#### Importing Packages
For this progect, I use <b>pandas</b> and <b>numpy</b> for data handling, <b>matplotlib</b> and <b>seaborn</b> for visualising, and <b>scikit-learn</b> for the multinomial model.

In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix)

#### Defining Variables Globally
From the very start, I want to be clearer about how we're processing the data. So for transparency, we're using the years 2018-2024. We chose 2024 because at the time of initial exploration, the 2025 session wasn't complete, and 2018 because the race style changes prior to the year, making analysis of those years unreliable. 

We further add a column to classify the years as pre-2022 and post-2022 to use for analysis of the regulation that was introduced in 2022. 

For race outcome, instead of just using the race finishing position from 1-20, we're doing it in categories for easier comparision. We decided on 3-class multinomial so first is positions 1-3 which we'll call "Podium", 4-10 as "Points", and 11-20 as "NoPoints". Another reason for this is smaple size per class, we want this to be more full which is why i've grouped classes together. changing this later, I think have it be 4-class (plus missing/dnf) is better with Winner, Podium, Points, NoPoints as the important classifications. 

We're using qualifying races and normal races which we'll denote the sessions as either "Q" for qualifying or "R". 

I'm also defining a random seed here to ensure reproducibility in the data splits and cross-validations. 

In [3]:
# defining years in analysis
START_YEAR = 2018
END_YEAR = 2024
YEARS = list(range(START_YEAR, END_YEAR + 1))

ERA_SPLIT_YEAR = 2022 # when regulation was introduced

# defining ttype of session
QUALI_SESSION = "Q"
RACE_SESSION = "R"

# definign outcome function
def define_outcome(race_pos):
    if pd.isna(race_pos):
        return np.nan
    elif race_pos <= 3:
        return "Podium"
    elif race_pos <= 10:
        return "Points"
    else:
        return "NoPoints"
    
# setting random seed
RANDOM_SEED = 5
np.random.seed(RANDOM_SEED)

# 2. Data Construction

In this section, I derive the dataset used for my multinomial logistic regression model. In this, I have one row as the Year, Round, Event, Driver --> Qualifying Performance --> Race Outcome, as that is the necessary informaiton for the model. 

# BELOW IS CGBT

2a. Extracting Official Race Schedules (No Hard-Coded Rounds)

Rather than manually defining race rounds, I extract the official event schedule directly from FastF1 for each year. This ensures:

Only valid race weekends are included

No missing or cancelled rounds are mistakenly referenced

The dataset automatically adapts to different season lengths

In [6]:
def get_race_schedule(year):
    """
    Returns a filtered race schedule for a given year,
    including only official Grand Prix events with race sessions.
    """
    schedule = fastf1.get_event_schedule(year)
    
    # Keep only conventional race weekends
    schedule = schedule[schedule["EventFormat"].isin(["conventional", "sprint"])]
    
    # Keep only actual races (exclude testing)
    schedule = schedule[~schedule["EventName"].str.contains("Test", case=False)]
    
    schedule = schedule.reset_index(drop=True)
    
    return schedule


# Build master schedule for all years
all_schedules = []

for year in YEARS:
    schedule_year = get_race_schedule(year)
    schedule_year["Year"] = year
    all_schedules.append(schedule_year)

schedule_df = pd.concat(all_schedules, ignore_index=True)

print("Total events extracted:", len(schedule_df))
schedule_df.head()


Total events extracted: 137


,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,...,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport,Year
0,1,Australia,Melbourne,FORMULA 1 2018 ROLEX AUSTRALIAN GRAND PRIX,2018-03-25,Australian Grand Prix,conventional,Practice 1,2018-03-23 12:00:00+11:00,2018-03-23 01:00:00,...,2018-03-24 14:00:00+11:00,2018-03-24 03:00:00,Qualifying,2018-03-24 17:00:00+11:00,2018-03-24 06:00:00,Race,2018-03-25 16:10:00+11:00,2018-03-25 05:10:00,True,2018
1,2,Bahrain,Sakhir,FORMULA 1 2018 GULF AIR BAHRAIN GRAND PRIX,2018-04-08,Bahrain Grand Prix,conventional,Practice 1,2018-04-06 14:00:00+03:00,2018-04-06 11:00:00,...,2018-04-07 15:00:00+03:00,2018-04-07 12:00:00,Qualifying,2018-04-07 18:00:00+03:00,2018-04-07 15:00:00,Race,2018-04-08 18:10:00+03:00,2018-04-08 15:10:00,True,2018
2,3,China,Shanghai,FORMULA 1 2018 HEINEKEN CHINESE GRAND PRIX,2018-04-15,Chinese Grand Prix,conventional,Practice 1,2018-04-13 10:00:00+08:00,2018-04-13 02:00:00,...,2018-04-14 11:00:00+08:00,2018-04-14 03:00:00,Qualifying,2018-04-14 14:00:00+08:00,2018-04-14 06:00:00,Race,2018-04-15 14:10:00+08:00,2018-04-15 06:10:00,True,2018
3,4,Azerbaijan,Baku,FORMULA 1 2018 AZERBAIJAN GRAND PRIX,2018-04-29,Azerbaijan Grand Prix,conventional,Practice 1,2018-04-27 13:00:00+04:00,2018-04-27 09:00:00,...,2018-04-28 14:00:00+04:00,2018-04-28 10:00:00,Qualifying,2018-04-28 17:00:00+04:00,2018-04-28 13:00:00,Race,2018-04-29 16:10:00+04:00,2018-04-29 12:10:00,True,2018
4,5,Spain,Barcelona,FORMULA 1 GRAN PREMIO DE ESPAÑA EMIRATES 2018,2018-05-13,Spanish Grand Prix,conventional,Practice 1,2018-05-11 11:00:00+02:00,2018-05-11 09:00:00,...,2018-05-12 12:00:00+02:00,2018-05-12 10:00:00,Qualifying,2018-05-12 15:00:00+02:00,2018-05-12 13:00:00,Race,2018-05-13 15:10:00+02:00,2018-05-13 13:10:00,True,2018


2b. Constructing the Event–Driver Dataset

In this section, I extract:

Qualifying session results (Q)

Race session results (R)

Merge them at the driver level

Add derived labels for:

<b>Era</b>

<b>Circuit Type</b>

<b>Race Outcome</b>

🏎️ Define Circuit Type Mapping (For Q1a)

Circuit types are grouped into broad categories reflecting overtaking difficulty.

You can refine this later if needed.


Circuits were categorised (PASSIVE) into Street, High-Speed, and Technical groups based on track layout characteristics (permanent vs temporary), average speed, and overtaking profile, using circuit guides from Formula1.com and FIA documentation. This grouping is an analytical simplification intended to proxy overtaking difficulty. should I make this into a different dataset and then import it?

In [18]:
# -------------------------------------------------------
# Circuit Type Mapping
# -------------------------------------------------------
# Categories:
# - "Street": narrow, limited overtaking (e.g., Monaco)
# - "HighSpeed": long straights / very fast layouts
# - "Technical": balanced permanent circuits (default)
# -------------------------------------------------------

CIRCUIT_TYPE_MAP = {

    # --- STREET ---
    "Monaco Grand Prix": "Street",
    "Singapore Grand Prix": "Street",
    "Azerbaijan Grand Prix": "Street",
    "Las Vegas Grand Prix": "Street",
    "Miami Grand Prix": "Street",
    "Australian Grand Prix": "Street",      # semi-street (Albert Park)
    "Canadian Grand Prix": "Street",        # semi-street (Gilles Villeneuve)

    # --- HIGH SPEED ---
    "Italian Grand Prix": "HighSpeed",
    "Belgian Grand Prix": "HighSpeed",
    "Saudi Arabian Grand Prix": "HighSpeed",
    "British Grand Prix": "HighSpeed",
    "Austrian Grand Prix": "HighSpeed",
    "Mexican Grand Prix": "HighSpeed",
    "São Paulo Grand Prix": "HighSpeed",
    "Brazilian Grand Prix": "HighSpeed",
    "Japanese Grand Prix": "HighSpeed",
    "Dutch Grand Prix": "HighSpeed",
    "Qatar Grand Prix": "HighSpeed",

    # --- TECHNICAL ---
    "Bahrain Grand Prix": "Technical",
    "Chinese Grand Prix": "Technical",
    "Spanish Grand Prix": "Technical",
    "Hungarian Grand Prix": "Technical",
    "French Grand Prix": "Technical",
    "German Grand Prix": "Technical",
    "United States Grand Prix": "Technical",
    "Emilia Romagna Grand Prix": "Technical",
    "Portuguese Grand Prix": "Technical",
    "Turkish Grand Prix": "Technical",
    "Abu Dhabi Grand Prix": "Technical"   # Yas Marina is permanent
}


def assign_circuit_type(event_name):
    """
    Assigns a circuit category based on event name.
    Defaults to 'Technical' if not found.
    """
    if isinstance(event_name, str):
        event_name = event_name.strip()
    return CIRCUIT_TYPE_MAP.get(event_name, "Technical")


print("Circuit type mapping defined correctly.")


Circuit type mapping defined correctly.


🏁 Extract Qualifying + Race Data Per Event

In [19]:
def get_event_driver_data(year, round_number, event_name):
    """
    Returns merged qualifying and race results for one event (one row per driver).

    Key design choice:
    - We load *only* what we need for results-level analysis.
      This dramatically reduces API calls and avoids hitting the 500 calls/hour limit.
    """
    try:
        # --- Load Qualifying (lightweight) ---
        quali = fastf1.get_session(year, round_number, QUALI_SESSION)
        quali.load(telemetry=False, laps=False, weather=False)

        # If results are missing, skip safely
        if quali.results is None or len(quali.results) == 0:
            print(f"Skipping {year} Round {round_number} ({event_name}): Qualifying results empty")
            return None

        q_results = quali.results.copy()

        # Best available qualifying time: Q3 if exists else Q2 else Q1
        q_results["QualiTime"] = (
            q_results["Q3"].fillna(q_results["Q2"]).fillna(q_results["Q1"])
        )

        q_keep = q_results[[
            "DriverNumber",
            "Abbreviation",
            "FullName",
            "TeamName",
            "Position",
            "Q1", "Q2", "Q3",   # keep raw sessions (optional but useful)
            "QualiTime"
        ]].rename(columns={"Position": "QualiPos"})

        # --- Load Race (lightweight) ---
        race = fastf1.get_session(year, round_number, RACE_SESSION)
        race.load(telemetry=False, laps=False, weather=False)

        if race.results is None or len(race.results) == 0:
            print(f"Skipping {year} Round {round_number} ({event_name}): Race results empty")
            return None

        r_results = race.results.copy()

        r_keep = r_results[[
            "DriverNumber",
            "Abbreviation",
            "Position",
            "GridPosition",
            "Status"
        ]].rename(columns={
            "Position": "RacePos",
            "GridPosition": "StartPos"
        })

        # --- Merge ---
        df = pd.merge(
            q_keep,
            r_keep,
            on=["DriverNumber", "Abbreviation"],
            how="inner"
        )

        # If merge drops everyone, skip (rare but can happen with weird identifiers)
        if len(df) == 0:
            print(f"Skipping {year} Round {round_number} ({event_name}): Merge produced 0 rows")
            return None

        # --- Add Metadata ---
        df["Year"] = year
        df["Round"] = round_number
        df["EventName"] = event_name

        # Era label
        df["Era"] = np.where(df["Year"] < ERA_SPLIT_YEAR, "Pre-2022", "Post-2022")

        # Circuit type
        df["CircuitType"] = df["EventName"].apply(assign_circuit_type)

        # Outcome (3-class)
        df["Outcome"] = df["RacePos"].apply(define_outcome)

        # Basic sanity: cast key positions to numeric (FastF1 often uses floats)
        for col in ["QualiPos", "RacePos", "StartPos"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        return df

    except Exception as e:
        print(f"Skipping {year} Round {round_number} ({event_name}) due to error: {repr(e)}")
        return None


🔁 Loop Through All Events
My comment: there is an api limit of 500 per hour so you have to run the api several times across different hours. This is also why i'm extracting the minimum amount of information so it's minimal reruns. Using cache, I can store the information so I don't reuse the API calls on the same data. For reproducibility and anyone else who would like to check witht he data, I've saveed the loaded data into a csv so they don't have to go through the hassle of running the code multiple times to get the data. 

In [27]:
event_driver_data = []

for idx, row in schedule_df.iterrows():
    year = row["Year"]
    round_number = row["RoundNumber"]
    event_name = row["EventName"]
    
    print(f"Processing {year} - Round {round_number} - {event_name}")
    
    df_event = get_event_driver_data(year, round_number, event_name)
    
    if df_event is not None:
        event_driver_data.append(df_event)

# Combine into final dataset
event_driver_df = pd.concat(event_driver_data, ignore_index=True)

print("\nFinal dataset shape:", event_driver_df.shape)
event_driver_df.head()


Processing 2018 - Round 1 - Australian Grand Prix


core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '7', '5', '33', '3', '20', '8', '27', '55', '77', '14', '2', '11', '18', '31', '28', '9', '16', '35', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '31', '16', '18', '28', '8', '20', '10', '9', '35']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req    

Processing 2018 - Round 2 - Bahrain Grand Prix


req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '7', '77', '44', '3', '10', '20', '27', '31', '55', '28', '11', '14', '2', '33', '8', '9', '35', '16', '18']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '77', '44', '10', '20', '27', '14', '2', '9', '31', '55', '16', '8', '18', '35', '11', '28', '7', '33', '3']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Processing 2018 - Round 3 - Chinese Grand Prix


...

req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '55', '10', '16', '1', '4', '22', '81', '27', '44', '31', '20', '24', '43', '30', '11', '14', '23', '77', '18']
core           INFO 	Loading data for Las Vegas Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
Request for URL https://api.jolpi.ca/ergast/f1/2024/22/results.json failed; using cached response
Traceback (most recent call last):
  File "/Users/minaheelkhan/anaconda3/lib/python3.11/site-packages/requests_cache/session.py", line 297, in _resend
    response.raise_for_status()
  File "/Users/minaheelkhan/anaconda3/lib/python3.11/site-packages/requests/models.py", line 1021, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/22/resul

Processing 2024 - Round 24 - Abu Dhabi Grand Prix


req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '55', '27', '1', '10', '63', '14', '77', '11', '22', '30', '18', '16', '20', '23', '24', '44', '43', '61']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
Request for URL https://api.jolpi.ca/ergast/f1/2024/24/results.json failed; using cached response
Traceback (most recent call last):
  File "/Users/minaheelkhan/anaconda3/lib/python3.11/site-packages/requests_cache/session.py", line 297, in _resend
    response.raise_for_status()
  File "/Users/minaheelkhan/anaconda3/lib/python3.11/site-packages/requests/models.py", line 1021, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/24/resul


Final dataset shape: (2738, 18)


,DriverNumber,Abbreviation,FullName,TeamName,QualiPos,Q1,Q2,Q3,QualiTime,RacePos,StartPos,Status,Year,Round,EventName,Era,CircuitType,Outcome
0,44,HAM,Lewis Hamilton,Mercedes,1.0,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,0 days 00:01:21.164000,2.0,1.0,Finished,2018,1,Australian Grand Prix,Pre-2022,Street,Podium
1,7,RAI,Kimi Räikkönen,Ferrari,2.0,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,0 days 00:01:21.828000,3.0,2.0,Finished,2018,1,Australian Grand Prix,Pre-2022,Street,Podium
2,5,VET,Sebastian Vettel,Ferrari,3.0,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,0 days 00:01:21.838000,1.0,3.0,Finished,2018,1,Australian Grand Prix,Pre-2022,Street,Podium
3,33,VER,Max Verstappen,Red Bull Racing,4.0,0 days 00:01:23.483000,0 days 00:01:22.416000,0 days 00:01:21.879000,0 days 00:01:21.879000,6.0,4.0,Finished,2018,1,Australian Grand Prix,Pre-2022,Street,Points
4,3,RIC,Daniel Ricciardo,Red Bull Racing,5.0,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000,0 days 00:01:22.152000,4.0,8.0,Finished,2018,1,Australian Grand Prix,Pre-2022,Street,Points


2c. Data Cleaning and Validation Checks

Before proceeding to EDA and modelling, I verify:

No duplicate keys

Outcome distribution is sensible

Race positions fall within expected range

🔍 Check for Duplicate Driver-Event Rows

In [10]:
duplicate_check = event_driver_df.duplicated(
    subset=["Year", "Round", "Abbreviation"]
).sum()

print("Duplicate rows:", duplicate_check)


Duplicate rows: 0


📊 Inspect Outcome Distribution

In [11]:
event_driver_df["Outcome"].value_counts()


NoPoints    200
Points      140
Podium       60
Name: Outcome, dtype: int64

🏎️ Inspect Basic Statistics

In [12]:
event_driver_df[["QualiPos", "RacePos", "StartPos"]].describe()


,QualiPos,RacePos,StartPos
count,400.000000,400.000000,400.000000
mean,10.500000,10.500000,10.450000
std,5.773503,5.773503,5.777625
min,1.000000,1.000000,0.000000
25%,5.750000,5.750000,5.000000
50%,10.500000,10.500000,10.000000
75%,15.250000,15.250000,15.000000
max,20.000000,20.000000,20.000000


2d. Save Dataset for Reproducibility

Saving the dataset ensures:

Results can be replicated without re-downloading

The R time-series analysis can import the same clean data

In [ ]:
event_driver_df.to_csv("event_driver_df.csv", index=False)

print("Dataset saved as event_driver_df.csv")


In [28]:
event_driver_df["Year"].value_counts().sort_index()


2018    420
2019    420
2020    340
2021    439
2022    440
2023    320
2024    359
Name: Year, dtype: int64

# after second run through 
2018    420
2019    420
2020    340
2021    439
2022    439
Name: Year, dtype: int64

# after third run through
2018    420
2019    420
2020    340
2021    439
2022    439
2023    320
2024    359
Name: Year, dtype: int64

# same after fourth run through so yay done